In [ ]:
import rdkit
import pandas as pdasdasdas
import numpy as np
import requests
from bs4 import BeautifulSoasdasdsadsadup


ModuleNotFoundError: No module named 'BeautifulSoup'

In [ ]:
!pip install bs4

In [17]:
def extrair_dados_medchemexpress(url):
    """
    Acessa uma página de produto do MedChemExpress e extrai o código HY (Cat. No.) e a string SMILES.
    Retorna um dicionário com os resultados.
    """
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    # Estrutura inicial do resultado
    resultados = {
        "hy_code": None,
        "smiles": None
    }

    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status() 
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # ==========================================
        # PASSO 1: Extrair o Código HY (Cat. No.)
        # ==========================================
        # Procura a label "Cat. No" ignorando maiúsculas, minúsculas e pontuações
        cat_no_label = soup.find(string=re.compile(r'Cat\.\s*No', re.IGNORECASE))
        
        if cat_no_label:
            container_cat = cat_no_label.parent.parent
            texto_cat = container_cat.get_text(separator=" ", strip=True)
            # Busca o padrão HY- seguido de letras ou números
            match_hy = re.search(r'(HY-[A-Z0-9]+)', texto_cat)
            if match_hy:
                resultados["hy_code"] = match_hy.group(1)
        
        # Fallback de Segurança para o Código HY:
        # Se a label "Cat. No." não for encontrada, varre o texto puro da página em busca do primeiro código HY- válido
        if not resultados["hy_code"]:
            match_fallback = re.search(r'(HY-[A-Z0-9]+)', soup.get_text())
            if match_fallback:
                resultados["hy_code"] = match_fallback.group(1)


        # ==========================================
        # PASSO 2: Extrair o SMILES (Sua lógica original)
        # ==========================================
        smiles_label = soup.find(string=re.compile(r'SMILES', re.IGNORECASE))
        
        if smiles_label:
            parent_tag = smiles_label.parent
            container = parent_tag.parent
            
            texto_completo = container.get_text(separator=" ", strip=True)
            smiles_str = re.sub(r'^.*?SMILES\s*:?\s*', '', texto_completo, flags=re.IGNORECASE).strip()
            
            if not smiles_str or len(smiles_str) < 3: 
                proximo_irmao = parent_tag.find_next_sibling()
                if proximo_irmao:
                    smiles_str = proximo_irmao.get_text(strip=True)
            
            resultados["smiles"] = smiles_str

        return resultados['hy_code'], resultados['smiles']

    except requests.exceptions.RequestException as e:
        print(f"Erro de requisição para {url}: {e}")
        return resultados

# Exemplo de uso:
# dados = extrair_dados_medchemexpress("https://www.medchemexpress.com/ipatasertib.html")
# print(f"Código: {dados['hy_code']} | SMILES: {dados['smiles']}")

,kinase,Compound,IC50 [M] ePCA,KD eq [M] ePCA *,KD kin [M] kPCA,KD [M] PCA **,kon [M-1s-1] kPCA,koff [s-1] kPCA
0,ABL,3-Methyladenine,"2,00E-05",NaN,"2,00E-05","2,00E-05",NaN,NaN
1,ABL,A66,"2,00E-05",NaN,"2,00E-05","2,00E-05",NaN,NaN
2,ABL,A-674563,"2,00E-05",NaN,"2,00E-05","2,00E-05",NaN,NaN
3,ABL,A-769662,"2,00E-05",NaN,"2,00E-05","2,00E-05",NaN,NaN
4,ABL,AEE788 (NVP-AEE788),"1,79E-06","1,42E-08","1,01E-08","1,01E-08","1,65E+07","1,55E-01"
...,...,...,...,...,...,...,...,...
10555,TRKB,Y-27632 2HCl,"2,00E-05",NaN,"2,00E-05","2,00E-05",NaN,NaN
10556,TRKB,YM201636,"2,00E-05",NaN,"2,00E-05","2,00E-05",NaN,NaN
10557,TRKB,ZM 336372,"2,00E-05",NaN,"2,00E-05","2,00E-05",NaN,NaN
10558,TRKB,ZM-447439,"2,00E-05",NaN,"5,26E-07","5,26E-07","6,65E+05","3,50E-01"


In [13]:
df = pd.read_csv('kinases.csv')
df = df[['kinase', 'Compound', 'KD eq [M] ePCA *']]
df.columns = ['prot', 'lig', 'kd']
df['kd'] = df['kd'].str.replace(',','.')
df['kd'] = pd.to_numeric(df['kd'], errors='coerce')
df = df.dropna()
df

,prot,lig,kd
4,ABL,AEE788 (NVP-AEE788),1.420000e-08
7,ABL,AG-1478 (Tyrphostin AG-1478),1.400000e-07
24,ABL,AT9283,1.930000e-08
25,ABL,Aurora A Inhibitor I,1.350000e-07
26,ABL,Axitinib,1.380000e-08
...,...,...,...
10533,TRKB,TWS119,8.900000e-07
10539,TRKB,VX-680 (MK-0457 - Tozasertib),2.380000e-08
10549,TRKB,WZ3146,1.160000e-07
10551,TRKB,WZ8040,1.240000e-07


In [14]:
df['pkd'] = -np.log10(df['kd'])
df = df.loc[df['pkd']>=9]
df = df.sort_values(by='pkd', ascending=False)
df

,prot,lig,kd,pkd
81,ABL,Dasatinib (BMS-354825),1.530000e-12,11.815309
930,AurA,MK-5108 (VX-689),4.610000e-12,11.336299
187,ABL,Ponatinib (AP24534),5.140000e-12,11.289037
2455,EGFR,Dacomitinib (PF299804 - PF-00299804),1.340000e-11,10.872895
1149,AXL,Foretinib (GSK1363089 - XL880),1.380000e-11,10.860121
...,...,...,...,...
9973,TGFBR1,SB 525334,9.890000e-10,9.004804
6190,KIT,Ki8751,9.960000e-10,9.001741
6115,KIT,BIBF1120 (Vargatef),9.960000e-10,9.001741
6302,KIT,Tivozanib (AV-951),9.960000e-10,9.001741


In [15]:
lst_lig = list(set(df['lig'].tolist()))

df = pd.DataFrame(lst_lig)
df.columns = ['name']
df

,name
0,Linsitinib (OSI-906)
1,Bosutinib (SKI-606)
2,AMG-208
3,AG-1478 (Tyrphostin AG-1478)
4,VX-680 (MK-0457 - Tozasertib)
...,...
84,Dasatinib (BMS-354825)
85,Hesperadin
86,AZD8931
87,Ruxolitinib (INCB018424)


In [25]:

try:
    with open("kinase_ligs_unique_mols.json", "r") as file:
        d = json.load(file)
except:
    d={}    

    i=1
    for name,link in pd.read_csv('links.csv').values:
        d[name] = {}

        hy, smi = extrair_dados_medchemexpress(link)
        print(i, link, hy, smi)
        #time.sleep(2)
        
        d[name]['link'] = link
        d[name]['smiles'] = smi
        d[name]['hy'] = hy

        i += 1
    
    with open("kinase_ligs_unique_mols.json", "w") as file:
        json.dump(d, file)

d

NameError: name 'BeautifulSoup' is not defined